## ✅ 오즈비, 카이제곱, Cramer's V 계산(3분화 버전 파일로 계산)
------------------------------------------------------------
### 목표
 1) stats/pivots.py: long df -> (필터/조건 반영) 교차표(tab/pivot) 생성만 담당: stats.pivots
 2) 교차표로부터 로그오즈비 계산만 담당: from stats.logodds
 3) 교차표로부터 카이제곱/크래머V 계산만 담당: from stats.chi2_utils


### 두 방식으로 돌아감.
 - (A) context × outcome (multi-level outcome) : tab (RxC)

    : make_context_outcome_tab, logodds_context_by_outcome_from_tab, chi2_overall_from_tab
    

 - (B) unit × context × binary-outcome : pivot (unit별 2xK + unit×context 2x2 log-odds)

    : make_unit_context_binary_pivot, logodds_unit_by_context_from_pivot, chi2_by_unit_from_pivot


In [12]:
# ------------------------------------------------------------------
# ✅ 사용 패턴 예시 (너의 기존 함수들을 "조합"으로 대체)
# ------------------------------------------------------------------
"""
# (A) context × outcome (multi-level)
from stats.pivots import make_context_outcome_tab
from stats.logodds import logodds_context_by_outcome_from_tab
from stats.chi2_utils import chi2_overall_from_tab

tab, ctx_vals, out_vals, mode = make_context_outcome_tab(
    df,
    context_col="category",
    outcome_col="EP_T",
    count_mode="weight",
    weight_col="ID",
    filters={...},
    context_top_n=10,
    outcome_top_n=30,
)

result_df = logodds_context_by_outcome_from_tab(
    tab,
    context_col_name="category",
    outcome_col_name="outcome_level",#tab의 컬럼 이름이 outcome_level이므로 이 이름을 기본값으로 설정함.
    alpha=0.5,
    add_alpha_if_zero=True,
    count_mode=mode,
)

chi2_df = chi2_overall_from_tab(tab, count_mode=mode)


# (B) unit × context × binary outcome (동사별 장르차)
from stats.pivots import make_unit_context_binary_pivot
from stats.logodds import logodds_unit_by_context_from_pivot
from stats.chi2_utils import chi2_by_unit_from_pivot

pivot, unit_vals, ctx_vals, mode = make_unit_context_binary_pivot(
    df,
    unit_col="V_No",
    context_col="category",
    outcome_col="EP_T",
    positive_fn=default_binary_parser, # True/False 변환 함수 예) positive_fn = lambda x: "었" in str(x)
    count_mode="weight",
    weight_col="ID",
    filters={...},
    unit_top_n=1000,
)

logodds_df = logodds_unit_by_context_from_pivot(
    pivot,
    unit_col_name="V_No",
    context_col_name="category",
    alpha=0.5,
    add_alpha_if_zero=True,
    count_mode=mode,
)

chi2_unit_df = chi2_by_unit_from_pivot(
    pivot,
    unit_col_name="V_No",
    count_mode=mode,
)
"""


'\n# (A) context × outcome (multi-level)\nfrom stats.pivots import make_context_outcome_tab\nfrom stats.logodds import logodds_context_by_outcome_from_tab\nfrom stats.chi2_utils import chi2_overall_from_tab\n\ntab, ctx_vals, out_vals, mode = make_context_outcome_tab(\n    df,\n    context_col="category",\n    outcome_col="EP_T",\n    count_mode="weight",\n    weight_col="ID",\n    filters={...},\n    context_top_n=10,\n    outcome_top_n=30,\n)\n\nresult_df = logodds_context_by_outcome_from_tab(\n    tab,\n    context_col_name="category",\n    outcome_col_name="outcome_level",#tab의 컬럼 이름이 outcome_level이므로 이 이름을 기본값으로 설정함.\n    alpha=0.5,\n    add_alpha_if_zero=True,\n    count_mode=mode,\n)\n\nchi2_df = chi2_overall_from_tab(tab, count_mode=mode)\n\n\n# (B) unit × context × binary outcome (동사별 장르차)\nfrom stats.pivots import make_unit_context_binary_pivot\nfrom stats.logodds import logodds_unit_by_context_from_pivot\nfrom stats.chi2_utils import chi2_by_unit_from_pivot\n\npivot, unit_val

In [13]:
import sys
from pathlib import Path

ROOT = Path.cwd().parents[0]
sys.path.append(str(ROOT))

from stats.pivots import make_context_outcome_tab, make_unit_context_binary_pivot #이 함수들은 피벗 테이블을 만들어주는 함수로, logodds나 chi2 계산 전에 데이터를 준비하는 단계에서 사용됨.
from stats.logodds import logodds_context_by_outcome_from_tab, logodds_unit_by_context_from_pivot #이 함수들은 피벗 테이블을 입력으로 받아서 logodds 계산을 수행하는 함수로, 실제로 logodds 계산을 하는 단계에서 사용됨.
from stats.chi2_utils import chi2_overall_from_tab, chi2_by_unit_from_pivot #이 함수들은 피벗 테이블을 입력으로 받아서 카이제곱 검정을 수행하는 함수로, 실제로 chi2 계산을 하는 단계에서 사용됨.


In [3]:
from datetime import datetime
import pandas as pd
import numpy as np

CSV_PATH = r"..\csv\통계용\세종_문어_문장끝_인용제외_20260405_17-55.csv"
COUNT_MODE = "weight" #가중치 파일
WEIGHT_COL="count" #입력 파일이 이미 가중치가 적용된 형태이므로, 가중치 컬럼을 지정하여 "개수" 대신 "가중치 합계"로 계산하도록 함.

df = pd.read_csv(CSV_PATH)

print(f"CSV 파일 로드 완료: {len(df):,}, now: {datetime.now().strftime("%Y.%m.%d_%H:%M:%S")}")
df.columns

C:\Users\yu2hy\AppData\Local\Temp\ipykernel_12428\2525599071.py:9: DtypeWarning: Columns (37,38,42) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV_PATH)


CSV 파일 로드 완료: 1,038,556, now: 2026.04.09_14:33:57


Index(['Unnamed: 0', 'category', '매체', 'file_id', 'docu_id', 'speaker',
       'sen_count', 'sen_count_has_E', 'sen_count_not_quote',
       'sen_count_has_E_not_quote', 'base_count_not_quote', 'dominant_EN_No',
       'dominant_EN_label', 'dominant_count', 'dominant_ratio', 'V_No',
       'V_form', 'V_label', 'EP_form', 'EN_form', 'EN_label', 'EN_No',
       'EN_No_sub', 'VX_len', 'Next_VX_No', 'Next_VX_form', 'VX0_No',
       'VX0_form', 'VX0_order', 'V0_form', 'V0_No', 'V0_label', 'f_EP_form',
       'f_EN_form', 'f_EN_No', 'f_EN_No_sub', 'f_EN_label',
       'has_prev_sentence', 'has_next_sentence', 'sentence_f_EP_form',
       'prev_sentence_f_EP_form', 'next_sentence_f_EP_form',
       'sentence_sent_end_V_in_quote', 'prev_sentence_sent_end_V_in_quote',
       'next_sentence_sent_end_V_in_quote', 'EP_TT', 'EP_T', 'EP_M', 'f_EP_TT',
       'f_EP_T', 'f_EP_M', 'sentence_f_EP_TT', 'sentence_f_EP_T',
       'sentence_f_EP_M', 'prev_sentence_f_EP_TT', 'prev_sentence_f_EP_T',
       'p

In [4]:
df['dominant_EN_No'].value_counts()
#df.loc[df['dominant_EN_No'].isna(), 'category'].value_counts()

dominant_EN_No
 1101.0    953146
 1171.0     52764
 1401.0     14806
-1.0        13713
 1411.0      1325
 1271.0      1147
 3002.0       456
 1221.0       373
 1451.0       156
 1331.0       136
 1301.0       102
 1999.0        75
 1179.0        60
 1241.0        55
 1231.0        38
 1202.0        34
 3001.0        34
 1372.0        26
 1431.0        23
 701.0         20
 1167.0        19
 1373.0        11
 1105.0         6
 1409.0         6
 10.0           6
 70.0           3
 1421.0         3
 20.0           2
 1432.0         2
 999.0          2
 90.0           2
 110.0          2
 25.0           1
 151.0          1
Name: count, dtype: int64

In [5]:
#document 파일 읽기
DOCU_CSV = r"..\csv\original_csv\세종문어_document_정보_ver.1.2.csv"
df_docu = pd.read_csv(DOCU_CSV, low_memory=False)

print(f"파일 읽기 완료 - {datetime.now().strftime('%Y.%m.%d_%H:%M:%S')}")
print(f"df_docu: {df_docu.columns.tolist()}")


파일 읽기 완료 - 2026.04.09_14:34:09
df_docu: ['docu_id', 'file_id', 'docu_num', 'category', '매체', '내용', '내용2', '파일제목', '저자', '출판사', '출판연도', 'head', '제목', '구어/문어', '분류기호', '분류기호2', '내용3', '분류기호4', 'docu_sen_count', 'docu_sen_count_has_E', 'docu_sen_count_not_quote', 'docu_sen_count_has_E_not_quote', 'docu_base_count_not_quote', 'docu_dominant_EN_No', 'docu_dominant_EN_label', 'docu_dominant_count', 'docu_dominant_ratio', 'docu_sent_count', 'docu_head_count', 'docu_body_count', 'docu_body_has_E_count', 'docu_body_not_quote_count', 'docu_body_not_quote_and_었_count', 'docu_었_결합_오즈비', 'docu_었_결합_로그오즈비', 'docu_었_결합_등급', 'docu_었_결합_성향', 'category2', 'file_sent_count', 'file_head_count', 'file_body_count', 'file_body_has_E_count', 'file_body_not_quote_count', 'file_body_not_quote_and_었_count', 'file_었_결합_오즈비', 'file_었_결합_로그오즈비', 'file_었_결합_등급', 'file_었_결합_성향']


In [6]:
df_docu['category'].value_counts()


category
보도해설    14729
인문사회    10265
사설       1631
자연       1517
체험기술     1469
칼럼       1357
허구일반     1229
총류        727
허구아동      231
Name: count, dtype: int64

In [7]:
df.drop(columns="category", inplace=True)  

In [8]:
# ---
# document 정보에서 필요한 컬럼 병합
# ---
def safe_merge(df, df_add, key, cols):
    cols_to_use = [key] + [c for c in cols if c not in df.columns]
    return df.merge(df_add[cols_to_use], on=key, how="left")

cols = [c for c in df_docu.columns if c != "file_id"]
df = safe_merge(df, df_docu, "docu_id", cols)

print(f"merge 완료 - {datetime.now().strftime('%Y.%m.%d_%H:%M:%S')}")
print(df.columns.to_list())

merge 완료 - 2026.04.09_14:34:17
['Unnamed: 0', '매체', 'file_id', 'docu_id', 'speaker', 'sen_count', 'sen_count_has_E', 'sen_count_not_quote', 'sen_count_has_E_not_quote', 'base_count_not_quote', 'dominant_EN_No', 'dominant_EN_label', 'dominant_count', 'dominant_ratio', 'V_No', 'V_form', 'V_label', 'EP_form', 'EN_form', 'EN_label', 'EN_No', 'EN_No_sub', 'VX_len', 'Next_VX_No', 'Next_VX_form', 'VX0_No', 'VX0_form', 'VX0_order', 'V0_form', 'V0_No', 'V0_label', 'f_EP_form', 'f_EN_form', 'f_EN_No', 'f_EN_No_sub', 'f_EN_label', 'has_prev_sentence', 'has_next_sentence', 'sentence_f_EP_form', 'prev_sentence_f_EP_form', 'next_sentence_f_EP_form', 'sentence_sent_end_V_in_quote', 'prev_sentence_sent_end_V_in_quote', 'next_sentence_sent_end_V_in_quote', 'EP_TT', 'EP_T', 'EP_M', 'f_EP_TT', 'f_EP_T', 'f_EP_M', 'sentence_f_EP_TT', 'sentence_f_EP_T', 'sentence_f_EP_M', 'prev_sentence_f_EP_TT', 'prev_sentence_f_EP_T', 'prev_sentence_f_EP_M', 'next_sentence_f_EP_TT', 'next_sentence_f_EP_T', 'next_sentence

In [9]:
df['문서범주'] = df['category'].map({
    '보도해설':'신문', 
    '사설': '신문', 
    '칼럼': '신문', 
    '인문사회': '책', 
    '체험기술': '체험', 
    '허구일반': '허구', 
    '자연': '책', 
    '총류': '책', 
    '허구아동': '허구'
})
df['문서범주'].value_counts()


문서범주
책     375024
허구    317423
신문    256468
체험     89641
Name: count, dtype: int64

In [10]:
df["시제_유지"] = df["sentence_f_EP_T"] == df["prev_sentence_f_EP_T"]

In [14]:
# (A) context × outcome (multi-level)
from datetime import datetime
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")

MOAD = "A"
CONTEXT_COL="docu_id"
OUTCOME_COL="sentence_f_EP_T" 


FILTER_T_SWITCH_PREV = {
    "speaker": "body",
    'has_prev_sentence': True,
    'sentence_sent_end_V_in_quote': False,
    'prev_sentence_sent_end_V_in_quote': False,
}
FILTER_1101 = {
    'f_EN_No': 1101,
    'f_EN_label': "EF",
}
FILTER_1101_DOCU = {  
    "speaker": "body",  
    "dominant_EN_No": 1101,
    'sen_count_has_E_not_quote': lambda x: x > 9}

FILTER_V  = {"VX0_No": -1}
FILTER_F    =  {"VX_len": 0}
FILTER_QUOTE = {'sentence_sent_end_V_in_quote': False}

FILTER_T_SWITCH_NEXT = {
    "speaker": "body",
    'has_next_sentence': True, 
    'sentence_sent_end_V_in_quote': False,
    'next_sentence_sent_end_V_in_quote': False
}

tab, ctx_vals, out_vals, mode = make_context_outcome_tab(
    df,
    context_col=CONTEXT_COL,
    outcome_col=OUTCOME_COL,
    count_mode=COUNT_MODE,
    weight_col=WEIGHT_COL,
    filters=FILTER_1101 | FILTER_F | FILTER_QUOTE,
    #context_top_n=30,
    #outcome_top_n=30,
)

result_df = logodds_context_by_outcome_from_tab(
    tab,
    context_col_name=CONTEXT_COL,
    alpha=0.5,
    add_alpha_if_zero=True,
    count_mode=mode,
)

chi2_df = chi2_overall_from_tab(tab, count_mode=mode)
print(f"[OK] made pivot, logodds, chi2: {OUTCOME_COL}_by_{CONTEXT_COL}, 실행시간: {timestamp}")


[OK] made pivot, logodds, chi2: sentence_f_EP_T_by_docu_id, 실행시간: 2026-04-09_14-35


In [30]:
# =========================================================
# 4. save to file
# =========================================================


#---- file name settings ----  
SAVE_DIR = Path("..") / "csv"/"결과값"/"tense"

out_file_oz = SAVE_DIR / f'{MOAD}_oz_{OUTCOME_COL}_by_{CONTEXT_COL}_{timestamp}.csv'
out_file_chi2 = SAVE_DIR / f'{MOAD}_chi2_{OUTCOME_COL}_by_{CONTEXT_COL}_{timestamp}.csv'
out_pivot = SAVE_DIR / f'{MOAD}_pivot_{OUTCOME_COL}_by_{CONTEXT_COL}_{timestamp}.csv'

print(f"***저장합니다.:    {datetime.now()}***")
tab.to_csv(out_pivot,index = False, encoding="utf-8-sig")
print(f"Output file for pivot table: {out_pivot}")

result_df.to_csv(out_file_oz, index=False, encoding="utf-8-sig")
print(f"[OK] saved logodds_{out_file_oz}")

if chi2_df is not None:
    chi2_df.to_csv(out_file_chi2, index=False, encoding="utf-8-sig")
    print(f"[OK] saved chi2_{out_file_chi2}")

***저장합니다.:    2026-04-02 21:11:08.472217***
Output file for pivot table: ..\csv\결과값\tense\A_pivot_sentence_f_EP_T_by_docu_id_2026-04-02_21-10.csv
[OK] saved logodds_..\csv\결과값\tense\A_oz_sentence_f_EP_T_by_docu_id_2026-04-02_21-10.csv
[OK] saved chi2_..\csv\결과값\tense\A_chi2_sentence_f_EP_T_by_docu_id_2026-04-02_21-10.csv


In [86]:
df['category'].unique()

array(['보도해설', '사설', '칼럼', '인문사회', '체험기술', '허구일반', '자연', '총류', '허구아동'],
      dtype=object)

In [ ]:
# (B) unit × context × binary outcome 
MOAD = "B"
UNIT_COL="docu_id"
CONTEXT_COL='prev_sentence_f_EP_T'
OUTCOME_COL="sentence_f_EP_T"
FILTER_T_SWITCH_PREV = {
    "speaker": "body",
    'has_prev_sentence': True,
    'sentence_sent_end_V_in_quote': False,
    'prev_sentence_sent_end_V_in_quote': False,
}
FILTER_1101 = {
    'f_EN_No': 1101,
    'f_EN_label': "EF",
}
FILTER_V  = {"VX0_No": -1}
FILTER_F    =  {"VX_len": 0}
FILTER_QUOTE = {'sentence_sent_end_V_in_quote': False}

FILTER_T_SWITCH_NEXT = {
    "speaker": "body",
    'has_next_sentence': True, 
    'sentence_sent_end_V_in_quote': False,
    'next_sentence_sent_end_V_in_quote': False
}
from stats.pivots import make_unit_context_binary_pivot
from stats.logodds import logodds_unit_by_context_from_pivot
from stats.chi2_utils import chi2_by_unit_from_pivot

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")

pivot, unit_vals, ctx_vals, mode = make_unit_context_binary_pivot(
    df,
    unit_col=UNIT_COL,
    context_col=CONTEXT_COL,
    outcome_col=OUTCOME_COL,
    #positive_fn=default_binary_parser,
    count_mode=COUNT_MODE,
    weight_col=WEIGHT_COL,
    filters=FILTER_1101 | FILTER_F | FILTER_QUOTE,
    #unit_top_n=1000,
)
print(f"[OK] made pivot: {UNIT_COL}X{CONTEXT_COL}_{OUTCOME_COL}, 실행시간: {timestamp}")


[OK] made pivot: docu_idXprev_sentence_f_EP_T_sentence_f_EP_T, 실행시간: 2026-04-02_14-29


In [27]:
# =========================================================
# Mode B, make logodds and save to file
# =========================================================

logodds_df = logodds_unit_by_context_from_pivot(
    pivot,
    unit_col_name=UNIT_COL,
    context_col_name=CONTEXT_COL,
    alpha=0.5,
    add_alpha_if_zero=True,
    count_mode=mode,
)

chi2_unit_df = chi2_by_unit_from_pivot(
    pivot,
    unit_col_name=UNIT_COL,
    count_mode=mode,
)

#---- file name settings ----  
SAVE_DIR = Path("..") / "csv"/"결과값"/"tense"
out_pivot = SAVE_DIR / f'{MOAD}_pivot_{UNIT_COL}_{OUTCOME_COL}_by_{CONTEXT_COL}_{timestamp}.csv'
out_file_oz = SAVE_DIR / f'{MOAD}_oz_{UNIT_COL}_{OUTCOME_COL}_by_{CONTEXT_COL}_{timestamp}.csv'
out_file_chi2 = SAVE_DIR / f'{MOAD}_chi2_{UNIT_COL}_{OUTCOME_COL}_by_{CONTEXT_COL}_{timestamp}.csv'

print(f"***저장합니다.:    {datetime.now()}***")
pivot_reset = pivot.reset_index()
pivot_reset.to_csv(out_pivot, index=False, encoding="utf-8-sig")
print(f"[OK] saved pivot_{out_pivot}")
logodds_df.to_csv(out_file_oz, index=False, encoding="utf-8-sig")
print(f"[OK] saved logodds_{out_file_oz}")

if chi2_unit_df is not None:
    chi2_unit_df.to_csv(out_file_chi2, index=False, encoding="utf-8-sig")
    print(f"[OK] saved chi2_{out_file_chi2}")

***저장합니다.:    2026-04-02 14:30:10.047014***
[OK] saved pivot_..\csv\결과값\tense\B_pivot_docu_id_sentence_f_EP_T_by_prev_sentence_f_EP_T_2026-04-02_14-29.csv
[OK] saved logodds_..\csv\결과값\tense\B_oz_docu_id_sentence_f_EP_T_by_prev_sentence_f_EP_T_2026-04-02_14-29.csv
[OK] saved chi2_..\csv\결과값\tense\B_chi2_docu_id_sentence_f_EP_T_by_prev_sentence_f_EP_T_2026-04-02_14-29.csv


In [ ]:
# =========================================================
# 5. (Optional) Exact runs distribution and summarizer
# =========================================================
# This section provides functions to compute the exact distribution of runs in a binary sequence and to summarize weighted transition data by unit, including permutation-test p-values based on the runs distribution.

import math
from typing import Optional, Tuple, Any

import numpy as np
import pandas as pd


# ============================================================
# 1. Small helpers
# ============================================================

def _normalize_binary_state(x: Any) -> Optional[bool]:
    """
    Normalize many possible T/F encodings to True/False.

    Accepted True-like values:
        True, 1, "1", "T", "TRUE", "Y", "YES"
    Accepted False-like values:
        False, 0, "0", "F", "FALSE", "N", "NO"

    Returns
    -------
    bool or None
        None means missing/unknown.
    """
    if pd.isna(x):
        return None

    if isinstance(x, (bool, np.bool_)):
        return bool(x)

    if isinstance(x, (int, np.integer, float, np.floating)):
        if x == 1:
            return True
        if x == 0:
            return False

    s = str(x).strip().upper()
    true_set = {"1", "T", "TRUE", "Y", "YES"}
    false_set = {"0", "F", "FALSE", "N", "NO"}

    if s in true_set:
        return True
    if s in false_set:
        return False

    raise ValueError(f"Could not normalize binary state: {x!r}")


def _safe_div(num: float, den: float) -> float:
    return np.nan if den == 0 else num / den


def _two_sided_p_from_z(z: float) -> float:
    """
    Two-sided p-value from standard normal using math.erfc.
    """
    if pd.isna(z):
        return np.nan
    return math.erfc(abs(z) / math.sqrt(2))


# ============================================================
# 2. Exact runs distribution
# ============================================================

def runs_pmf(n_t: int, n_f: int) -> pd.DataFrame:
    """
    Exact PMF of the number of runs R for a binary sequence with fixed counts:
        n_t = number of T states
        n_f = number of F states

    References: standard Wald-Wolfowitz runs distribution.

    Returns
    -------
    DataFrame with columns:
        R, k, prob
    where k = R - 1 = number of alternations
    """
    n_t = int(n_t)
    n_f = int(n_f)

    if n_t < 0 or n_f < 0:
        raise ValueError("n_t and n_f must be nonnegative.")
    if n_t + n_f == 0:
        return pd.DataFrame(columns=["R", "k", "prob"])

    total = math.comb(n_t + n_f, n_t)
    rows = []

    r_min = 1 if (n_t == 0 or n_f == 0) else 2
    r_max = 1 if (n_t == 0 or n_f == 0) else 2 * min(n_t, n_f) + (1 if n_t != n_f else 0)

    for r in range(r_min, r_max + 1):
        if r % 2 == 0:
            # r = 2m
            m = r // 2
            if m - 1 <= n_t - 1 and m - 1 <= n_f - 1 and m >= 1:
                ways = 2 * math.comb(n_t - 1, m - 1) * math.comb(n_f - 1, m - 1)
            else:
                ways = 0
        else:
            # r = 2m + 1
            m = (r - 1) // 2
            ways = 0
            # T-start / T-end pattern
            if m <= n_t - 1 and m - 1 <= n_f - 1 and m >= 1:
                ways += math.comb(n_t - 1, m) * math.comb(n_f - 1, m - 1)
            elif m == 0 and n_f == 0 and n_t > 0:
                ways += 1

            # F-start / F-end pattern
            if m - 1 <= n_t - 1 and m <= n_f - 1 and m >= 1:
                ways += math.comb(n_t - 1, m - 1) * math.comb(n_f - 1, m)
            elif m == 0 and n_t == 0 and n_f > 0:
                ways += 1

        if ways > 0:
            rows.append((r, r - 1, ways / total))

    out = pd.DataFrame(rows, columns=["R", "k", "prob"])
    if not out.empty:
        out["prob"] = out["prob"] / out["prob"].sum()
    return out


def runs_exact_p_values(k_obs: int, n_t: int, n_f: int) -> Tuple[float, float, float]:
    """
    Exact p-values for the observed number of alternations k_obs.

    Returns
    -------
    (p_lower, p_upper, p_two_sided)

    p_lower:
        P(K <= k_obs)
    p_upper:
        P(K >= k_obs)
    p_two_sided:
        exact two-sided p based on distance from E[K]
        i.e. sum of probabilities with |K - E[K]| >= |k_obs - E[K]|
    """
    pmf = runs_pmf(n_t, n_f)
    if pmf.empty:
        return (np.nan, np.nan, np.nan)

    e_k = 2 * n_t * n_f / (n_t + n_f)

    p_lower = pmf.loc[pmf["k"] <= k_obs, "prob"].sum()
    p_upper = pmf.loc[pmf["k"] >= k_obs, "prob"].sum()

    obs_dist = abs(k_obs - e_k)
    p_two = pmf.loc[(pmf["k"] - e_k).abs() >= obs_dist - 1e-12, "prob"].sum()

    return (p_lower, p_upper, min(1.0, p_two))


# ============================================================
# 3. Core summarizer from weighted transition rows
# ============================================================

def summarize_weighted_transitions(
    df: pd.DataFrame,
    *,
    unit_col: str,
    context_col: str,
    outcome_col: str,
    count_col: str = "count",
    # Option A: directly provide per-unit state counts
    state_counts_df: Optional[pd.DataFrame] = None,
    n_t_col: str = "n_T",
    n_f_col: str = "n_F",
    # Option B: provide per-unit first/last states so n_T/n_F can be derived
    first_last_df: Optional[pd.DataFrame] = None,
    first_state_col: str = "first_state",
    last_state_col: str = "last_state",
    # Smoothing for log OR
    add_half_for_log_or: bool = True,
) -> pd.DataFrame:
    """
    Summarize weighted transition rows by unit.

    Input format
    ------------
    Each row is a weighted transition count:
        unit_col, context_col, outcome_col, count_col

    Example:
        docu_id, prev_sentence_f_EP_T, sentence_f_EP_T, count

    Required for permutation test
    -----------------------------
    You must provide either:
    - state_counts_df with n_T and n_F per unit, OR
    - first_last_df with first_state and last_state per unit

    Returns
    -------
    DataFrame with one row per unit and columns including:
        a_TT, b_TF, c_FT, d_FF
        n_pairs
        k_alt
        E_k, Var_k, SD_k, z_k, p_norm_two_sided
        p_exact_lower, p_exact_upper, p_exact_two_sided
        Pearson residuals
        odds_ratio, log_odds_ratio
    """
    work = df.copy()

    # Normalize binary states
    work["_prev"] = work[context_col].map(_normalize_binary_state)
    work["_curr"] = work[outcome_col].map(_normalize_binary_state)

    if work["_prev"].isna().any() or work["_curr"].isna().any():
        raise ValueError("Some context/outcome states could not be normalized to binary.")

    # Pivot weighted 2x2 counts by unit
    g = (
        work.groupby([unit_col, "_prev", "_curr"], dropna=False)[count_col]
        .sum()
        .rename("count")
        .reset_index()
    )

    pivot = (
        g.pivot_table(
            index=unit_col,
            columns=["_prev", "_curr"],
            values="count",
            aggfunc="sum",
            fill_value=0,
        )
        .copy()
    )

    def _get_col(prev_state: bool, curr_state: bool) -> pd.Series:
        if (prev_state, curr_state) in pivot.columns:
            return pivot[(prev_state, curr_state)].astype(float)
        return pd.Series(0.0, index=pivot.index)

    out = pd.DataFrame(index=pivot.index)
    out["a_TT"] = _get_col(True, True)
    out["b_TF"] = _get_col(True, False)
    out["c_FT"] = _get_col(False, True)
    out["d_FF"] = _get_col(False, False)

    out["n_pairs"] = out[["a_TT", "b_TF", "c_FT", "d_FF"]].sum(axis=1)

    # Transition summaries
    out["k_alt"] = out["b_TF"] + out["c_FT"]
    out["R_runs"] = out["k_alt"] + 1

    out["prev_T_total"] = out["a_TT"] + out["b_TF"]
    out["prev_F_total"] = out["c_FT"] + out["d_FF"]
    out["curr_T_total"] = out["a_TT"] + out["c_FT"]
    out["curr_F_total"] = out["b_TF"] + out["d_FF"]

    out["alt_rate_pairs"] = out["k_alt"] / out["n_pairs"]
    out["TF_rate_given_prev_T"] = out["b_TF"] / out["prev_T_total"]
    out["FT_rate_given_prev_F"] = out["c_FT"] / out["prev_F_total"]

    # ------------------------------------------------------------
    # Recover or merge n_T / n_F
    # ------------------------------------------------------------
    if state_counts_df is not None:
        state_counts = state_counts_df[[unit_col, n_t_col, n_f_col]].copy()
        out = out.reset_index().merge(state_counts, on=unit_col, how="left").set_index(unit_col)

    elif first_last_df is not None:
        fl = first_last_df[[unit_col, first_state_col, last_state_col]].copy()
        fl["_first"] = fl[first_state_col].map(_normalize_binary_state)
        fl["_last"] = fl[last_state_col].map(_normalize_binary_state)

        out = out.reset_index().merge(
            fl[[unit_col, "_first", "_last"]],
            on=unit_col,
            how="left",
        ).set_index(unit_col)

        # Derive n_T / n_F from transition table + last/first state
        out["n_T_from_last"] = out["a_TT"] + out["b_TF"] + out["_last"].fillna(False).astype(int)
        out["n_F_from_last"] = out["c_FT"] + out["d_FF"] + (~out["_last"].fillna(True)).astype(int)

        out["n_T_from_first"] = out["a_TT"] + out["c_FT"] + out["_first"].fillna(False).astype(int)
        out["n_F_from_first"] = out["b_TF"] + out["d_FF"] + (~out["_first"].fillna(True)).astype(int)

        out[n_t_col] = out["n_T_from_last"]
        out[n_f_col] = out["n_F_from_last"]

        # Consistency check
        bad = (
            (out["n_T_from_last"] != out["n_T_from_first"]) |
            (out["n_F_from_last"] != out["n_F_from_first"])
        )
        if bad.any():
            bad_units = out.index[bad].tolist()
            raise ValueError(
                "Could not derive consistent n_T / n_F from first/last state for these units: "
                f"{bad_units[:10]}{'...' if len(bad_units) > 10 else ''}"
            )
    else:
        # No state counts available
        out[n_t_col] = np.nan
        out[n_f_col] = np.nan

    # ------------------------------------------------------------
    # Permutation-test quantities
    # ------------------------------------------------------------
    n = out[n_t_col] + out[n_f_col]
    out["n_states"] = n

    out["E_k"] = 2 * out[n_t_col] * out[n_f_col] / n
    out["E_TF"] = out[n_t_col] * out[n_f_col] / n
    out["E_FT"] = out["E_TF"]

    # Var(K) = Var(R)
    out["Var_k"] = (
        2 * out[n_t_col] * out[n_f_col] *
        (2 * out[n_t_col] * out[n_f_col] - n)
    ) / (n.pow(2) * (n - 1))

    out.loc[n <= 1, "Var_k"] = np.nan
    out["SD_k"] = np.sqrt(out["Var_k"])
    out["z_k"] = (out["k_alt"] - out["E_k"]) / out["SD_k"]
    out["p_norm_two_sided"] = out["z_k"].map(_two_sided_p_from_z)

    # Exact runs p-values
    p_lower_list = []
    p_upper_list = []
    p_two_list = []

    for _, row in out.iterrows():
        if pd.isna(row[n_t_col]) or pd.isna(row[n_f_col]):
            p_lower_list.append(np.nan)
            p_upper_list.append(np.nan)
            p_two_list.append(np.nan)
            continue

        p_lower, p_upper, p_two = runs_exact_p_values(
            int(row["k_alt"]),
            int(row[n_t_col]),
            int(row[n_f_col]),
        )
        p_lower_list.append(p_lower)
        p_upper_list.append(p_upper)
        p_two_list.append(p_two)

    out["p_exact_lower"] = p_lower_list
    out["p_exact_upper"] = p_upper_list
    out["p_exact_two_sided"] = p_two_list

    # ------------------------------------------------------------
    # Pearson residuals for the 2x2 table
    # ------------------------------------------------------------
    N = out["n_pairs"]

    out["E_a_TT"] = (out["prev_T_total"] * out["curr_T_total"]) / N
    out["E_b_TF"] = (out["prev_T_total"] * out["curr_F_total"]) / N
    out["E_c_FT"] = (out["prev_F_total"] * out["curr_T_total"]) / N
    out["E_d_FF"] = (out["prev_F_total"] * out["curr_F_total"]) / N

    out["pearson_a_TT"] = (out["a_TT"] - out["E_a_TT"]) / np.sqrt(out["E_a_TT"])
    out["pearson_b_TF"] = (out["b_TF"] - out["E_b_TF"]) / np.sqrt(out["E_b_TF"])
    out["pearson_c_FT"] = (out["c_FT"] - out["E_c_FT"]) / np.sqrt(out["E_c_FT"])
    out["pearson_d_FF"] = (out["d_FF"] - out["E_d_FF"]) / np.sqrt(out["E_d_FF"])

    # ------------------------------------------------------------
    # Odds ratio / log odds ratio
    # ------------------------------------------------------------
    if add_half_for_log_or:
        a = out["a_TT"] + 0.5
        b = out["b_TF"] + 0.5
        c = out["c_FT"] + 0.5
        d = out["d_FF"] + 0.5
    else:
        a, b, c, d = out["a_TT"], out["b_TF"], out["c_FT"], out["d_FF"]

    out["odds_ratio"] = (a * d) / (b * c)
    out["log_odds_ratio"] = np.log(out["odds_ratio"])

    # Nice final order
    preferred_cols = [
        "a_TT", "b_TF", "c_FT", "d_FF",
        "n_pairs", "n_states", n_t_col, n_f_col,
        "k_alt", "R_runs",
        "alt_rate_pairs", "TF_rate_given_prev_T", "FT_rate_given_prev_F",
        "E_k", "E_TF", "E_FT",
        "Var_k", "SD_k", "z_k",
        "p_norm_two_sided", "p_exact_lower", "p_exact_upper", "p_exact_two_sided",
        "pearson_a_TT", "pearson_b_TF", "pearson_c_FT", "pearson_d_FF",
        "odds_ratio", "log_odds_ratio",
    ]
    existing_cols = [c for c in preferred_cols if c in out.columns]
    other_cols = [c for c in out.columns if c not in existing_cols]
    out = out[existing_cols + other_cols].reset_index()

    return out


# ============================================================
# 4. Convenience wrapper for your column names
# ============================================================

def analyze_ep_transitions(
    df_weighted: pd.DataFrame,
    *,
    count_col: str = "count",
    unit_col: str = "docu_id",
    context_col: str = "prev_sentence_f_EP_T",
    outcome_col: str = "sentence_f_EP_T",
    state_counts_df: Optional[pd.DataFrame] = None,
    n_t_col: str = "n_T",
    n_f_col: str = "n_F",
    first_last_df: Optional[pd.DataFrame] = None,
    first_state_col: str = "first_state",
    last_state_col: str = "last_state",
) -> pd.DataFrame:
    """
    Thin wrapper using your variable names.
    """
    return summarize_weighted_transitions(
        df_weighted,
        unit_col=unit_col,
        context_col=context_col,
        outcome_col=outcome_col,
        count_col=count_col,
        state_counts_df=state_counts_df,
        n_t_col=n_t_col,
        n_f_col=n_f_col,
        first_last_df=first_last_df,
        first_state_col=first_state_col,
        last_state_col=last_state_col,
    )

In [ ]:
UNIT_COL = "docu_id"
CONTEXT_COL = "prev_sentence_f_EP_T"
OUTCOME_COL = "sentence_f_EP_T"

result = analyze_ep_transitions(
    df,
    count_col="count",
    unit_col=UNIT_COL,
    context_col=CONTEXT_COL,
    outcome_col=OUTCOME_COL,
    state_counts_df=df_state_counts,   # must contain: docu_id, n_T, n_F
    n_t_col="n_T",
    n_f_col="n_F",
)

print(result.head())